In [1]:
import pandas as pd
from absl import logging
import plotly.graph_objects as go
import sys
import math
sys.path.insert(0, './utils')

from amd_pcm_parsing import read_amd_pcm_out


In [11]:
def read_timestamps(ts_file):
    df = pd.read_csv(ts_file, parse_dates=[2], date_format='%H:%M:%S:%f')
    # time is from start of execution
    df['time'] = df['time'] - pd.Timestamp('1900-01-01')
    return df

baseline = True
pd_max = False
pd_dop4 = False
pd_stage_1 = True
pd_stage_both = True
basepath = "/scratch/nicholso/adm_output/gmi_x8_noavx_amd_pcm/"

if baseline:
    ts_baseline = read_timestamps(basepath + "sel_cpu_baseline/sel_cpu_baseline_timestamps.csv")
    df_baseline = read_amd_pcm_out(basepath + "sel_cpu_baseline/amd_pcm.csv")
else:
    ts_baseline = None
    df_baseline = None

if pd_max:
    ts_pd_max = read_timestamps(basepath + "sel_cpu_pd_dop_max/sel_cpu_pd_dop_max_timestamps.csv")
    df_pd_max = read_amd_pcm_out(basepath + "sel_cpu_pd_dop_max/amd_pcm.csv")
else:
    ts_pd_max = None
    df_pd_max = None

if pd_dop4:
    ts_pd_dop4 = read_timestamps(basepath + "sel_cpu_pd_dop_4/sel_cpu_pd_dop_4_timestamps.csv")
    df_pd_dop4 = read_amd_pcm_out(basepath + "sel_cpu_pd_dop_4/amd_pcm.csv")
else:
    ts_pd_dop4 = None
    df_pd_dop4 = None

if pd_stage_1:
    ts_pd_stage_1 = read_timestamps(basepath + "sel_cpu_stage_one/sel_cpu_stage_one_timestamps.csv")
    df_pd_stage_1 = read_amd_pcm_out(basepath + "sel_cpu_stage_one/amd_pcm.csv")
else:
    ts_pd_stage_1 = None
    df_pd_stage_1 = None

if pd_stage_both:
    ts_pd_stage_both = read_timestamps(basepath + "sel_cpu_stage_both/sel_cpu_stage_both_timestamps.csv")
    df_pd_stage_both = read_amd_pcm_out(basepath + "sel_cpu_stage_both/amd_pcm.csv")
else:
    ts_pd_stage_both = None
    df_pd_stage_both = None

len aggregations 61 len headers 62
len aggregations 61 len headers 62
len aggregations 61 len headers 62


In [12]:
from typing import Optional


def add_timestamp_start_end_bars(fig: go.Figure, df_timestamps: pd.DataFrame, bar_height : Optional[int] = None):
    if bar_height is None:
        if fig.layout.yaxis.range is not None:
            bar_height = fig.layout.yaxis.range[1]
        else:
            logging.info("Guessing figure height")
            bar_height = 10
    bar_height = round(0.9 * bar_height)
    for ts in df_timestamps.itertuples():
        if ts.point_type == 'start':
            start_ts_seconds = ts.time.total_seconds()
            # add line
            fig.add_shape(
                type="line",
                x0=start_ts_seconds, y0=0,  # start of the line
                x1=start_ts_seconds, y1=bar_height,  # end of the line
                line=dict(
                    color="green",
                    width=1,
                ),
                showlegend=False
            )
            # add tooltip
            fig.add_trace(go.Scatter(
                x=[start_ts_seconds, start_ts_seconds],
                y=[0, bar_height],
                mode='markers',
                marker=dict(
                    size=3,  # make marker size small so it's not visible
                    color="green"  # make marker color transparent
                ),
                hoverinfo='text',
                hovertext=[f"start {ts.label}", f"start {ts.label}"],
                showlegend=False
            ))

            matching_end_timestamps = df_timestamps[df_timestamps['point_type'].str.contains("end")] 
            matching_end_timestamps = matching_end_timestamps[matching_end_timestamps['label'].str.contains(ts.label)]
            if len(matching_end_timestamps) != 1:
                print(f"huh {len}")
            else:
                end_ts = matching_end_timestamps.iloc[0]
                end_ts_seconds = end_ts.time.total_seconds()
                # add end line
                fig.add_shape(
                    type="line",
                    x0=end_ts_seconds-0.5, y0=0,  # start of the line
                    x1=end_ts_seconds-0.5, y1=bar_height,  # end of the line
                    line=dict(
                        color="red",
                        width=1,
                    ),
                    showlegend=False
                )
                # add tooltip
                fig.add_trace(go.Scatter(
                    x=[end_ts_seconds-0.5, end_ts_seconds-0.5],
                    y=[0, bar_height],
                    mode='markers',
                    marker=dict(
                        size=3,
                        color="red"
                    ),
                    hoverinfo='text',
                    hovertext=[f"start {ts.label}", f"start {ts.label}"],
                    showlegend=False
                ))
    
def add_query_start_end_bars(fig: go.Figure, df_timestamps: pd.DataFrame):
    line_height = 0
    modulo_line_height = 15
    if fig.layout.yaxis.range is not None:
        modulo_line_height = fig.layout.yaxis.range[1] / 3.0
    line_height_add = modulo_line_height / 3.0
    for ts in df_timestamps.itertuples():
        if ts.point_type == 'start':
            start_ts_seconds = ts.time.total_seconds()
            
            matching_end_timestamps = df_timestamps[df_timestamps['point_type'].str.contains("end")] 
            matching_end_timestamps = matching_end_timestamps[matching_end_timestamps['label'].str.contains(ts.label)]
            end_ts = matching_end_timestamps[matching_end_timestamps['time'] > ts.time].min()

            end_ts_seconds = end_ts.time.total_seconds()
            
            line_height = (line_height % modulo_line_height) + line_height_add            

            # add line
            fig.add_shape(
                type="line",
                x0=start_ts_seconds, y0=line_height, 
                x1=end_ts_seconds, y1=line_height,
                line=dict(
                    color="black",
                    width=2,
                ),
                showlegend=False
            )
            # add tooltip
            fig.add_trace(go.Scatter(
                x=[start_ts_seconds, end_ts_seconds],
                y=[line_height, line_height],
                mode='markers',
                marker=dict(
                    size=3,  # make marker size small so it's not visible
                    color="black"  # make marker color transparent
                ),
                hoverinfo='text',
                hovertext=[f"start {ts.label}. Duration: {end_ts_seconds-start_ts_seconds:.2f}s", f"end {ts.label}. Duration: {end_ts_seconds-start_ts_seconds:.2f}s"],
                showlegend=False
            ))
            
import re
from typing import List
import plotly.express as px


class ColumnColourPallette:
    def __init__(self, columns: List[str]) -> None:
        self.unique_metrics = sorted(list(set([re.split(r'(-\d+)', val)[-1].strip() for val in columns])))
        # Find all the unique aggregates (not the entire string, just unique substr)
        matches = [re.findall(r'(-\d+)', val) for val in columns]
        self.unique_aggregates = sorted(list(set([match for sublist in matches for match in sublist])))
        base_color_scales = [px.colors.sequential.Reds, px.colors.sequential.Blues, px.colors.sequential.Greens, px.colors.sequential.Purples, px.colors.sequential.Oranges, px.colors.sequential.Teal, px.colors.sequential.ice, px.colors.sequential.Magenta]
        step_size = len(base_color_scales[0][:-1]) // len(self.unique_metrics)
        self.color_scales = [scale[:-1:step_size] for scale in base_color_scales]
        
    def __getitem__(self, column: str):
        metric_index = self.unique_metrics.index(re.split(r'(-\d+)', column)[-1].strip())
        agg_index = self.unique_aggregates.index(re.findall(r'(-\d+)', column)[0])
        return self.color_scales[agg_index][metric_index]

In [13]:



def total_mem_bw(data: pd.DataFrame, timestamps: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    max_y_bw: int = 120
    time_axis = data['Timestamp'].dt.total_seconds()
    y_values = sorted([col for col in data.columns if 'Total Mem Bw' in col])
    max_y = 0
    for y_val in y_values:
        max_y = max(max_y, data[y_val].max())
        fig.add_trace(go.Scatter(x=time_axis, y=data[y_val], mode='lines', name=y_val))

    fig.update_layout(title='Per socket memory BW utilization', xaxis_title='Time (seconds)', yaxis_title='Bandwidth (GB/s)')
    fig.update_yaxes(range=[0, max_y_bw])

    # start/end of warmup + iterations of specific selectivity
    df_timestamps = timestamps[timestamps['label'].str.contains("sel_")]
    add_timestamp_start_end_bars(fig, df_timestamps)
    df_query_timestamps = timestamps[timestamps['label'].str.contains("query_")]
    add_query_start_end_bars(fig, df_query_timestamps)
    return fig

if baseline:
    baseline_membw_fig = total_mem_bw(df_baseline, ts_baseline)
    baseline_membw_fig.update_layout(title='Per socket memory BW utilization (baseline)')
    baseline_membw_fig.show()

if pd_max:
    pd_max_membw_fig = total_mem_bw(df_pd_max, ts_pd_max)
    pd_max_membw_fig.update_layout(title='Per socket memory BW utilization (pd_max)')
    pd_max_membw_fig.show()

if pd_dop4:
    pd_dop4_membw_fig = total_mem_bw(df_pd_dop4, ts_pd_dop4)
    pd_dop4_membw_fig.update_layout(title='Per socket memory BW utilization (pd_dop4)')
    pd_dop4_membw_fig.show()


if pd_stage_1:
    pd_stage_1_membw_fig = total_mem_bw(df_pd_stage_1, ts_pd_stage_1)
    pd_stage_1_membw_fig.update_layout(title='Per socket memory BW utilization (pd_stage_1)')
    pd_stage_1_membw_fig.show()

if pd_stage_both:
    pd_stage_both_membw_fig = total_mem_bw(df_pd_stage_both, ts_pd_stage_both)
    pd_stage_both_membw_fig.update_layout(title='Per socket memory BW utilization (pd_stage_both)')
    pd_stage_both_membw_fig.show()


In [15]:
def xgmi_bw(data: pd.DataFrame, timestamps: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    max_y_bw = 30
    time_axis = data['Timestamp'].dt.total_seconds()
    y_values = sorted([col for col in data.columns if 'GMI' in col])
    col_pal = ColumnColourPallette(y_values)
    for y_val in y_values:
        fig.add_trace(go.Scatter(x=time_axis, y=data[y_val], mode='lines', name=y_val, line=dict(color=col_pal[y_val])))

    fig.update_layout(title='Per socket GMI BW utilization', xaxis_title='Time (seconds)', yaxis_title='Bandwidth (GB/s)')
    fig.update_yaxes(range=[0, max_y_bw])
    # start/end of warmup + iterations of specific selectivity
    df_timestamps = timestamps[timestamps['label'].str.contains("sel_")]
    add_timestamp_start_end_bars(fig, df_timestamps)
    df_query_timestamps = timestamps[timestamps['label'].str.contains("query_")]
    add_query_start_end_bars(fig, df_query_timestamps)
    return fig

# create all of the graphs
if baseline:
    baseline_xgmi_fig = xgmi_bw(df_baseline, ts_baseline)
    baseline_xgmi_fig.update_layout(title='Per socket GMI BW utilization (baseline)')
    baseline_xgmi_fig.show()

if pd_max:
    pd_max_xgmi_fig = xgmi_bw(df_pd_max, ts_pd_max)
    pd_max_xgmi_fig.update_layout(title='Per socket GMI BW utilization (pd_max)')
    pd_max_xgmi_fig.show()

if pd_dop4:
    pd_dop4_xgmi_fig = xgmi_bw(df_pd_dop4, ts_pd_dop4)
    pd_dop4_xgmi_fig.update_layout(title='Per socket GMI BW utilization (pd_dop4)')
    pd_dop4_xgmi_fig.show()
    
if pd_stage_1:
    pd_stage_1_xgmi_fig = xgmi_bw(df_pd_stage_1, ts_pd_stage_1)
    pd_stage_1_xgmi_fig.update_layout(title='Per socket GMI BW utilization (pd_stage_1)')
    pd_stage_1_xgmi_fig.show()

if pd_stage_both:
    pd_stage_both_xgmi_fig = xgmi_bw(df_pd_stage_both, ts_pd_stage_both)
    pd_stage_both_xgmi_fig.update_layout(title='Per socket GMI BW utilization (pd_stage_both)')
    pd_stage_both_xgmi_fig.show()

In [6]:
def dc_refills(data: pd.DataFrame, timestamps: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    

    start_stamps = timestamps[timestamps['point_type'].str.contains("start")] 
    start_bench_stamps = start_stamps[start_stamps['label'].str.contains("bench_")]
    first_bench_start = start_bench_stamps.min().time.total_seconds()

    bench_data = data[data['Timestamp'].dt.total_seconds() > first_bench_start]

    time_axis = data['Timestamp'].dt.total_seconds()

    y_values = sorted([col for col in data.columns if 'DC' in col and ('Memory' in col or 'All DC Fills' in col)])
    col_pal =  ColumnColourPallette(y_values)

    max_y = 0
    for y_val in y_values:
        max_y = max(max_y, bench_data[y_val].max())
        if 'Local' in y_val:
            dash = 'dot'
        if 'Remote' in y_val:
            dash = 'dash'
        else:
            dash = 'solid'
        fig.add_trace(go.Scatter(x=time_axis, y=data[y_val], mode='lines', name=y_val, line=dict(color=col_pal[y_val], dash=dash)))

    fig.update_layout(title='Per socket data cache fills', xaxis_title='Time (seconds)', yaxis_title='Fills (per thousand instructions)')
    fig.update_yaxes(range=[0, math.ceil(max_y/20) * 20])
    # start/end of warmup + iterations of specific selectivity
    df_timestamps = timestamps[timestamps['label'].str.contains("sel_")]
    add_timestamp_start_end_bars(fig, df_timestamps)
    df_query_timestamps = timestamps[timestamps['label'].str.contains("query_")]
    add_query_start_end_bars(fig, df_query_timestamps)

    fig.update_xaxes(range=[first_bench_start, data['Timestamp'].max().total_seconds()])
    return fig

if baseline:
    baseline_dc_fig = dc_refills(df_baseline, ts_baseline)
    baseline_dc_fig.update_layout(title='Per socket data cache fills (baseline)')
    baseline_dc_fig.show()

if pd_max:
    pd_max_dc_fig = dc_refills(df_pd_max, ts_pd_max)
    pd_max_dc_fig.update_layout(title='Per socket data cache fills (pd_max)')
    pd_max_dc_fig.show()

if pd_dop4:
    pd_dop4_dc_fig = dc_refills(df_pd_dop4, ts_pd_dop4)
    pd_dop4_dc_fig.update_layout(title='Per socket data cache fills (pd_dop4)')
    pd_dop4_dc_fig.show()

if pd_stage_1:
    pd_stage_1_dc_fig = dc_refills(df_pd_stage_1, ts_pd_stage_1)
    pd_stage_1_dc_fig.update_layout(title='Per socket data cache fills (pd_stage_1)')
    pd_stage_1_dc_fig.show()

if pd_stage_both:
    pd_stage_both_dc_fig = dc_refills(df_pd_stage_both, ts_pd_stage_both)
    pd_stage_both_dc_fig.update_layout(title='Per socket data cache fills (pd_stage_both)')
    pd_stage_both_dc_fig.show()
    

In [75]:
def software_prefetch_fills(data: pd.DataFrame, timestamps: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    

    start_stamps = timestamps[timestamps['point_type'].str.contains("start")] 
    start_bench_stamps = start_stamps[start_stamps['label'].str.contains("bench_")]
    first_bench_start = start_bench_stamps.min().time.total_seconds()

    bench_data = data[data['Timestamp'].dt.total_seconds() > first_bench_start]

    time_axis = data['Timestamp'].dt.total_seconds()

    y_values = sorted([col for col in data.columns if 'SwPf' in col and ('DRAM' in col or 'remote' in col) ])
    col_pal =  ColumnColourPallette(y_values)

    max_y = 0
    for y_val in y_values:
        max_y = max(max_y, bench_data[y_val].max())
        if 'Local' in y_val:
            dash = 'dot'
        if 'Remote' in y_val:
            dash = 'dash'
        else:
            dash = 'solid'
        fig.add_trace(go.Scatter(x=time_axis, y=data[y_val], mode='lines', name=y_val, line=dict(color=col_pal[y_val], dash=dash)))

    fig.update_layout(title='Per socket hardware cache prefetch fills', xaxis_title='Time (seconds)', yaxis_title='Fills (per thousand instructions)')
    fig.update_yaxes(range=[0, max_y])
    # start/end of warmup + iterations of specific selectivity
    df_timestamps = timestamps[timestamps['label'].str.contains("sel_")]
    add_timestamp_start_end_bars(fig, df_timestamps)
    df_query_timestamps = timestamps[timestamps['label'].str.contains("query_")]
    add_query_start_end_bars(fig, df_query_timestamps)

    fig.update_xaxes(range=[first_bench_start, data['Timestamp'].max().total_seconds()])
    return fig

if baseline:
    baseline_sft_prefetch_fig = software_prefetch_fills(df_baseline, ts_baseline)
    baseline_sft_prefetch_fig.update_layout(title='Per socket software cache prefetch fills (baseline)')
    baseline_sft_prefetch_fig.show()

if pd_max:
    pd_max_sft_prefetch_fig = software_prefetch_fills(df_pd_max, ts_pd_max)
    pd_max_sft_prefetch_fig.update_layout(title='Per socket software cache prefetch fills (pd_max)')
    pd_max_sft_prefetch_fig.show()

if pd_dop4:
    pd_dop4_sft_prefetch_fig = software_prefetch_fills(df_pd_dop4, ts_pd_dop4)
    pd_dop4_sft_prefetch_fig.update_layout(title='Per socket software cache prefetch fills (pd_dop4)')
    pd_dop4_sft_prefetch_fig.show()

if pd_stage_1:
    pd_stage_1_sft_prefetch_fig = software_prefetch_fills(df_pd_stage_1, ts_pd_stage_1)
    pd_stage_1_sft_prefetch_fig.update_layout(title='Per socket software cache prefetch fills (pd_stage_1)')
    pd_stage_1_sft_prefetch_fig.show()

if pd_stage_both:
    pd_stage_both_sft_prefetch_fig = software_prefetch_fills(df_pd_stage_both, ts_pd_stage_both)
    pd_stage_both_sft_prefetch_fig.update_layout(title='Per socket software cache prefetch fills (pd_stage_both)')
    pd_stage_both_sft_prefetch_fig.show()

In [73]:
def hardware_prefetch_fills(data: pd.DataFrame, timestamps: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    

    start_stamps = timestamps[timestamps['point_type'].str.contains("start")] 
    start_bench_stamps = start_stamps[start_stamps['label'].str.contains("bench_")]
    first_bench_start = start_bench_stamps.min().time.total_seconds()

    bench_data = data[data['Timestamp'].dt.total_seconds() > first_bench_start]

    time_axis = data['Timestamp'].dt.total_seconds()

    y_values = sorted([col for col in data.columns if 'HwPf' in col and ('DRAM' in col or 'remote' in col) ])
    col_pal =  ColumnColourPallette(y_values)

    max_y = 0
    for y_val in y_values:
        max_y = max(max_y, bench_data[y_val].max())
        if 'Local' in y_val:
            dash = 'dot'
        if 'Remote' in y_val:
            dash = 'dash'
        else:
            dash = 'solid'
        fig.add_trace(go.Scatter(x=time_axis, y=data[y_val], mode='lines', name=y_val, line=dict(color=col_pal[y_val], dash=dash)))

    fig.update_layout(title='Per socket hardware cache prefetch fills', xaxis_title='Time (seconds)', yaxis_title='Fills (per thousand instructions)')
    fig.update_yaxes(range=[0, max_y])
    # start/end of warmup + iterations of specific selectivity
    df_timestamps = timestamps[timestamps['label'].str.contains("sel_")]
    add_timestamp_start_end_bars(fig, df_timestamps)
    df_query_timestamps = timestamps[timestamps['label'].str.contains("query_")]
    add_query_start_end_bars(fig, df_query_timestamps)

    fig.update_xaxes(range=[first_bench_start, data['Timestamp'].max().total_seconds()])
    return fig

if baseline:
    baseline_hard_prefetch_fig = hardware_prefetch_fills(df_baseline, ts_baseline)
    baseline_hard_prefetch_fig.update_layout(title='Per socket hardware cache prefetch fills (baseline)')
    baseline_hard_prefetch_fig.show()

if pd_max:
    pd_max_hard_prefetch_fig = hardware_prefetch_fills(df_pd_max, ts_pd_max)
    pd_max_hard_prefetch_fig.update_layout(title='Per socket hardware cache prefetch fills (pd_max)')
    pd_max_hard_prefetch_fig.show()

if pd_dop4:
    pd_dop4_hard_prefetch_fig = hardware_prefetch_fills(df_pd_dop4, ts_pd_dop4)
    pd_dop4_hard_prefetch_fig.update_layout(title='Per socket hardware cache prefetch fills (pd_dop4)')
    pd_dop4_hard_prefetch_fig.show()

if pd_stage_1:
    pd_stage_1_hard_prefetch_fig = hardware_prefetch_fills(df_pd_stage_1, ts_pd_stage_1)
    pd_stage_1_hard_prefetch_fig.update_layout(title='Per socket hardware cache prefetch fills (pd_stage_1)')
    pd_stage_1_hard_prefetch_fig.show()

if pd_stage_both:
    pd_stage_both_hard_prefetch_fig = hardware_prefetch_fills(df_pd_stage_both, ts_pd_stage_both)
    pd_stage_both_hard_prefetch_fig.update_layout(title='Per socket hardware cache prefetch fills (pd_stage_both)')
    pd_stage_both_hard_prefetch_fig.show()

In [74]:
[col for col in df_pd_stage_both.columns if 'HwPf' in col]

['Package (Aggregated)-0 HwPf DC Fills From DRAM or IO connected in remote node (pti)',
 'Package (Aggregated)-0 HwPf DC Fills From CCX Cache in remote node (pti)',
 'Package (Aggregated)-0 HwPf DC Fills From DRAM or IO connected in local node (pti)',
 'Package (Aggregated)-0 HwPf DC Fills From Cache of another CCX in local node (pti)',
 'Package (Aggregated)-0 HwPf DC Fills From L3 or different L2 in same CCX (pti)',
 'Package (Aggregated)-0 HwPf DC Fills From L2 (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From DRAM or IO connected in remote node (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From CCX Cache in remote node (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From DRAM or IO connected in local node (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From Cache of another CCX in local node (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From L3 or different L2 in same CCX (pti)',
 'Package (Aggregated)-1 HwPf DC Fills From L2 (pti)']